
# Recurrent Neural Networks (RNN)

## 1. Introduction

Traditional Feedforward Neural Networks (ANNs) and CNNs have limitations when processing sequential data such as text, audio, and time-series.

**Major limitations:**
1. Fixed Input Length: Traditional ANNs generally require fixed-size input vectors.
2. Lack of Memory: They cannot naturally retain information from previous time steps.
3. No Temporal Dependency: They struggle to understand relationships between sequential elements.

**Recurrent Neural Networks (RNNs)** solve these problems using a recurrent hidden state that carries information from previous time steps.

### Key Features
- Processes data sequentially, one time step at a time.
- Maintains a hidden state (memory).
- Shares the same weights across all time steps.
- Can handle sequences of varying lengths.

---

## 2. Input Data Representation

RNNs require numerical input represented as a 3D tensor.

**Input Shape:**

`(Batch Size, Time Steps, Features)`

| Dimension | Meaning |
|---|---|
| Batch Size (N) | Number of sequences processed together |
| Time Steps (T) | Number of tokens/elements in each sequence |
| Features (D) | Number of features representing each element |

**Example:**

Consider a review containing 3 words and a vocabulary of 5 words.

Each word is represented using a 5-dimensional one-hot vector.

- Single sample shape: `(3, 5)`
- Batch input shape: `(Batch Size, 3, 5)`

---

## 3. RNN Architecture

An RNN processes input sequentially:

`x(1) → x(2) → x(3) → ... → x(T)`

At every time step, the current input is combined with the previous hidden state to calculate a new hidden state.

### Important Weight Matrices

| Weight | Function | Shape |
|---|---|---|
| Wi | Input to hidden state | (D, H) |
| Wh | Previous hidden state to current hidden state | (H, H) |
| Wo | Hidden state to output | (H, O) |

Where:
- D = Input Features
- H = Number of Hidden Units
- O = Output Neurons

**Parameter Sharing:** The same Wi and Wh matrices are reused at every time step. This reduces the number of trainable parameters.

### Initial Hidden State

The initial hidden state is generally initialized with zeros:

`h(0) = 0`

---

## 4. Forward Propagation in RNN

### Step 1: Calculate Hidden State

The current hidden state combines the current input and previous hidden state.

$
h^{(t)} = \tanh(x^{(t)}W_i + h^{(t-1)}W_h + b_h)
$

Where:
- x(t) = Current input
- h(t-1) = Previous hidden state
- Wi = Input weight matrix
- Wh = Recurrent weight matrix
- bh = Hidden layer bias
- tanh = Activation function

### Step 2: Calculate Output

For a sequence-to-label task, the final hidden state is passed to the output layer.

$
\hat{y} = f_{act}(h^{(T)}W_o + b_o)
$

Where:
- h(T) = Final hidden state
- Wo = Output weight matrix
- bo = Output bias
- f(act) = Output activation function

**Output Activation:**
- Sigmoid: Binary Classification
- Softmax: Multiclass Classification

---

## 5. Parameter Calculation

For a SimpleRNN layer:

$
Parameters = (D \times H) + (H \times H) + H
$

For a Dense output layer:

$
Parameters = (H \times O) + O
$

### Example

Given:
- Input Features (D) = 5
- Hidden Units (H) = 3
- Output Neurons (O) = 1

**SimpleRNN Parameters:**

$
(5 \times 3) + (3 \times 3) + 3 = 27
$

**Dense Layer Parameters:**

$
(3 \times 1) + 1 = 4
$

**Total Parameters:**

$
27 + 4 = \boxed{31}
$

---

## 6. Keras Implementation


In [1]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense

I0000 00:00:1794210771.647749     551 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794210771.975468     551 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794210773.814174     551 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
model = Sequential([
    Input(shape=(4, 5)),
    SimpleRNN(units=3, activation='tanh'),
    Dense(units=1, activation='sigmoid')
])

I0000 00:00:1794210780.940196     551 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [3]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [4]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 3)              │            27 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Expected Trainable Parameters: 31  


# RNN Sentiment Analysis

## 1. Introduction

Sentiment Analysis is a Natural Language Processing (NLP) task used to classify text based on its sentiment.

Example:
- "The movie was excellent" → Positive (1)
- "The movie was terrible" → Negative (0)

To process text using an RNN, raw sentences must first be converted into numerical sequences.

### Text Preprocessing Pipeline

`Raw Text → Tokenization → Integer Encoding → Padding → Input Tensor`

---

## 2. Tokenization

Tokenization converts text into individual words and creates a vocabulary dictionary.

**Keras Tokenizer:**
- Converts text to lowercase.
- Removes punctuation (based on its filters).
- Assigns a unique integer ID to each word.
- Creates a `word_index` dictionary.
- Supports Out-of-Vocabulary (OOV) tokens.

## 3. Integer Encoding

The `texts_to_sequences()` method converts each word into its corresponding integer ID.

Example:

```text
"I love machine learning"

"I"       → 1
"love"    → 2
"machine" → 3
"learning"→ 4

Encoded Sequence: [1, 2, 3, 4]
```

## 4. Sequence Padding

Different reviews have different lengths. RNN models generally require uniform sequence lengths within a batch.

The `pad_sequences()` function ensures that all sequences have the same length.

### Padding Types

| Type | Description |
|---|---|
| Pre-padding | Adds zeros before the sequence |
| Post-padding | Adds zeros after the sequence |
| Pre-truncating | Removes elements from the beginning |
| Post-truncating | Removes elements from the end |

Example:

```text
Original:       [1, 2, 3]
Post Padding:   [1, 2, 3, 0, 0]
Pre Padding:    [0, 0, 1, 2, 3]
```

**Important:** Padding ensures uniform sequence length, while truncation removes elements exceeding the specified maximum length.



# Workflow 1: Integer Encoding + SimpleRNN

## 1. Concept

In this approach, words are converted into integer IDs and passed directly to the RNN.

**Important:** Integer IDs are categorical identifiers, not meaningful numerical measurements. Passing them directly as continuous input features can introduce artificial numerical relationships between word IDs.

Therefore, this is mainly a basic demonstration approach rather than the recommended NLP architecture.

## 2. Keras Implementation


In [6]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb

In [7]:
# Hyperparameters
max_features = 10000
max_len = 50

# Load IMDB dataset
(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=max_features)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [8]:
# Padding
X_train = pad_sequences(
    X_train, maxlen=max_len,
    padding='post', truncating='post')

X_test = pad_sequences(
    X_test, maxlen=max_len,
    padding='post', truncating='post')

In [9]:
# Reshape: (Batch, Time Steps, Features)
X_train = X_train.reshape(-1, max_len, 1)
X_test = X_test.reshape(-1, max_len, 1)

In [10]:
# Build model
model = Sequential([
    Input(shape=(max_len, 1)),
    SimpleRNN(32, activation='tanh'),
    Dense(1, activation='sigmoid')])

In [11]:
# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Training
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/5


I0000 00:00:1794210973.721421     805 service.cc:153] XLA service 0x24cccb50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1794210973.721473     805 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.5.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.28.0)
I0000 00:00:1794210973.760148     805 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1794210973.917574     805 cuda_dnn.cc:461] Loaded cuDNN version 92800
I0000 00:00:1794210973.935535     805 dot_merger.cc:481] Merging Dots in computation: sequential_1_1_simple_rnn_1_1_while_body_1260_grad_1489_const_0__.17.clone.clone.clone.clone.clone


  8/782 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.5117 - loss: 1.1258

I0000 00:00:1794210975.197656     805 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


780/782 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5046 - loss: 0.7067

I0000 00:00:1794210984.893881     802 dot_merger.cc:481] Merging Dots in computation: sequential_1_1_simple_rnn_1_1_while_body_1260_grad_1489_const_0__.17.clone.clone.clone.clone.clone


782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.5044 - loss: 0.7067 - val_accuracy: 0.5033 - val_loss: 0.6946
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.4974 - loss: 0.6943 - val_accuracy: 0.5019 - val_loss: 0.6966
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.5013 - loss: 0.6940 - val_accuracy: 0.4995 - val_loss: 0.6946
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.5055 - loss: 0.6936 - val_accuracy: 0.5025 - val_loss: 0.6969
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.5018 - loss: 0.6939 - val_accuracy: 0.5016 - val_loss: 0.6951


# Workflow 2: Embedding + SimpleRNN (Recommended)

## 1. Concept

The Embedding layer converts integer word IDs into dense, low-dimensional vectors.

It acts as a trainable lookup table that learns word representations during model training.

### Advantages of Embedding

1. Dense Representation: Converts integer IDs into continuous vectors.
2. Trainable: Embedding weights are updated during backpropagation.
3. Dimensionality Reduction: Avoids explicitly creating large one-hot vectors.
4. Learned Relationships: Can capture useful similarities between words based on the training task.

### Embedding Layer Syntax

```python
Embedding(
    input_dim=vocabulary_size,
    output_dim=embedding_dimension,
    mask_zero=True
)
```

| Parameter | Meaning |
|---|---|
| input_dim | Vocabulary size |
| output_dim | Dimension of each word vector |
| mask_zero | Ignores padding tokens (ID 0) in supported layers |

### Input and Output Shapes

Input:

`(Batch Size, Time Steps)`

Output:

`(Batch Size, Time Steps, Embedding Dimension)`

## 2. Keras Implementation

In [14]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb

In [15]:
# Padding
X_train = pad_sequences(
    X_train, maxlen=max_len,
    padding='post', truncating='post'
)

X_test = pad_sequences(
    X_test, maxlen=max_len,
    padding='post', truncating='post'
)


In [17]:
# Hyperparameters
max_features = 10000
max_len = 50
embedding_dim = 32  # Dimension of each word embedding vector

In [18]:
# Build model
model = Sequential([
    Input(shape=(max_len,)),

    Embedding(
        input_dim=max_features,
        output_dim=embedding_dim,
        mask_zero=True
    ),

    SimpleRNN(
        units=32,
        activation='tanh',
        return_sequences=False
    ),

    Dense(1, activation='sigmoid')
])

In [19]:
# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 32)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 322,113 (1.23 MB)

 Trainable params: 322,113 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
# Training
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/5


I0000 00:00:1794211232.527697     803 dot_merger.cc:481] Merging Dots in computation: sequential_2_1_simple_rnn_2_1_while_body_35124_grad_35370_const_0__.17.clone.clone.clone.clone.clone.clone.clone.clone
I0000 00:00:1794211232.539130     803 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


386/391 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7111 - loss: 0.5586

I0000 00:00:1794211239.294848     805 dot_merger.cc:481] Merging Dots in computation: sequential_2_1_simple_rnn_2_1_while_body_35124_grad_35370_const_0__.17.clone.clone.clone.clone.clone.clone.clone.clone


391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.7116 - loss: 0.5578 - val_accuracy: 0.7524 - val_loss: 0.5074
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.8349 - loss: 0.3900 - val_accuracy: 0.7499 - val_loss: 0.5454
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9028 - loss: 0.2518 - val_accuracy: 0.7302 - val_loss: 0.6062
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.9623 - loss: 0.1147 - val_accuracy: 0.7295 - val_loss: 0.8157
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.9876 - loss: 0.0452 - val_accuracy: 0.6973 - val_loss: 0.9838


**Note:** `mask_zero=True` ensures that padding tokens are masked rather than treated as actual words. SimpleRNN supports masking.


# RNN: Parameter Calculation & Configurations

## 1. Parameter Calculation – Embedding + SimpleRNN

Given:

- Vocabulary Size (V) = 10,000
- Embedding Dimension (D) = 32
- Hidden Units (H) = 32
- Output Neurons (O) = 1

### Layer 1: Embedding

$
Parameters = V \times D
$

$
= 10000 \times 32 = 320000
$

### Layer 2: SimpleRNN

$
Parameters = (D \times H) + (H \times H) + H
$

$
= (32 \times 32) + (32 \times 32) + 32
$

$
= 2112
$

### Layer 3: Dense

$
Parameters = (H \times O) + O
$

$
= (32 \times 1) + 1 = 33
$

### Total Trainable Parameters

$
320000 + 2112 + 33 = \boxed{322145}
$

---

## 2. return_sequences Parameter

The `return_sequences` parameter determines what output is returned by the RNN layer.

### return_sequences=False (Default)

- Returns only the final hidden state.
- Output shape: `(Batch Size, Hidden Units)`.
- Suitable for sequence-to-label tasks.

**Example:** Sentiment Analysis.

### return_sequences=True

- Returns hidden states from all time steps.
- Output shape: `(Batch Size, Time Steps, Hidden Units)`.
- Useful for stacking recurrent layers and sequence-to-sequence tasks.

**Example:** Text generation and sequence labeling.

---

## 3. Types of RNN Input-Output Architectures

| Architecture | Input | Output | Example |
|---|---|---|---|
| One-to-One | Single | Single | Image Classification |
| One-to-Many | Single | Sequence | Image Captioning |
| Many-to-One | Sequence | Single | Sentiment Analysis |
| Many-to-Many | Sequence | Sequence | Machine Translation |

---

## 4. Important Takeaways

1. RNNs maintain memory through hidden states.
2. The same weights are shared across all time steps.
3. Hidden states carry information from previous time steps.
4. Padding makes sequences uniform in length.
5. Embedding converts integer IDs into learnable dense vectors.
6. `return_sequences=False` returns the final hidden state.
7. `return_sequences=True` returns the complete hidden-state sequence.
8. SimpleRNN can struggle with long-term dependencies due to vanishing gradients. LSTM and GRU help address this limitation.$